<a href="https://colab.research.google.com/github/lukeje/robot-example/blob/main/robotcar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pybullet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp312-cp312-linux_x86_64.whl size=99873445 sha256=0de44d9f3ae571b5818caf90cf152df77a65aa22227ad8c890dd729aa5d1c26e
  Stored in directory: /root/.cache/pip/wheels/72/95/1d/b336e5ee612ae9a019bfff4dc0bedd100ee6f0570db205fdf8
Successfully built pybullet
ERROR: Could not find a version that satisfies the requirement pybullet_data (from versions: none)
ERROR: No matching distribution found for pybullet_data


In [3]:
# adapted from https://github.com/akinami3/PybulletRobotics

import math
import time

import pybullet as pb
import pybullet_data
import controlcar

# Connect to the display
pb.connect(pb.DIRECT)


ModuleNotFoundError: No module named 'controlcar'

In [ ]:

# Set simulation parameters
pb.resetSimulation() # Reset the simulation space
pb.setAdditionalSearchPath(pybullet_data.getDataPath()) # Add path to necessary data for pybullet
pb.setGravity(0.0, 0.0, -9.8) # Set gravity as on Earth
timestep = 1.0/240.0
pb.setTimeStep(timestep) # Set the elapsed time per step
nsteps = 500 # How many steps taken for each motion

# Load the floor
plane_id = pb.loadURDF("plane.urdf")

# Set the camera position and other parameters in GUI mode
camera_distance = 2.0
camera_yaw = 0.0 # deg
camera_pitch = -20 # deg
camera_target_position = [0.0, 0.0, 0.0]
pb.resetDebugVisualizerCamera(camera_distance, camera_yaw, camera_pitch, camera_target_position)

# Load the robot
car_start_pos = [0, 0, 0.1] # Set the initial position (x, y, z)
car_start_orientation = pb.getQuaternionFromEuler([0, 0, 0]) # Set the initial orientation (roll, pitch, yaw)
car_id = pb.loadURDF("data/PybulletRobotics/urdf/simple_two_wheel_car.urdf", car_start_pos, car_start_orientation)

# Indices for joints
RIGHT_WHEEL_JOINT_IDX = 0
LEFT_WHEEL_JOINT_IDX  = 1

# Instantiate class to define the wheel velocities
mv = controlcar.MovementVelocities(timestep,nsteps)

# Loop over wheel velocities for different states
for v in (mv.goforwards(1), mv.gobackwards(0.5), mv.turnleft(math.pi/2), mv.turnright(math.pi)):
    # Set the target velocities for each wheel
    pb.setJointMotorControl2(car_id, RIGHT_WHEEL_JOINT_IDX, pb.VELOCITY_CONTROL, targetVelocity=v[RIGHT_WHEEL_JOINT_IDX])
    pb.setJointMotorControl2(car_id, LEFT_WHEEL_JOINT_IDX,  pb.VELOCITY_CONTROL, targetVelocity=v[LEFT_WHEEL_JOINT_IDX])

    # Step the simulation forwards in time
    for _ in range(nsteps):
        pb.stepSimulation()
        time.sleep(timestep)


In [ ]:
pb.disconnect()